# 문제해결 1: 금융사기 보고서 기반 Docling + Chroma RAG

## 실습 목적

Naive RAG의 기본 흐름을 익힌 후, **직접 문서 기반 질의응답 시스템을 구성하고 문제를 점검**하도록 설계한 실습 예제

사용 문서: `data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf`

## 실습 시나리오

금융소비자 교육 담당자가 하나의 금융사기 보고서를 기반으로 질문에 답변하는 내부용 QA 챗봇을 만들려고 한다.  

## 완성 기준

| 기준 | 설명 |
|---|---|
| 문서 로딩 | Docling으로 PDF를 변환하고 구조 기반 청크(`DOC_CHUNKS`)를 `Document` 객체로 정상 로딩한다. |
| 청킹 | `HybridChunker`로 문서 구조와 token 한도(700)에 맞게 청크가 생성된다. |
| 벡터DB 생성 | Chroma 컬렉션이 생성되고 로컬 저장까지 가능하다. |
| 검색 | 질문과 관련된 상위 문서 조각을 검색한다. |
| 답변 생성 | 검색된 문서에 근거해서만 답변한다. |

## 0. 설치

> `faiss-cpu`는 로컬 CPU 환경에서 FAISS를 사용하기 위한 패키지이다.

In [ ]:
# uv add langchain-docling langchain-chroma "docling-core[chunking-openai]" tiktoken python-dotenv pandas

## 1. 라이브러리 불러오기

LangChain의 기본 구성요소를 사용한다.

- `DoclingLoader`: Docling 변환 + 구조 기반 청크 로딩 (`ExportType.DOC_CHUNKS`)
- `HybridChunker`: 문서 구조 + token 기준 청킹
- `Chroma`: 로컬 벡터스토어
- `llm_connect()`: 답변 생성 LLM

In [ ]:
from pathlib import Path
import os
from pprint import pprint
import pandas as pd
import tiktoken

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import (
    HeadingHierarchyOptions,
    PdfPipelineOptions,
)
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.openai import OpenAITokenizer
from langchain_docling import DoclingLoader
from langchain_docling.loader import ExportType

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser


## 2. 환경변수 설정

LLM_API_KEY는 `.env` 파일 또는 운영체제 환경변수에 설정한다.

`.env` 파일 예시는 다음과 같다.

```text
LLM_API_KEY=mr_universal...
```

In [ ]:
# 환경 셋팅
import sys
sys.path.append("..")
from common_config import embedding_model, llm_connect
# from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# 임베딩 객체 생성
embeddings = embedding_model()

# 모델 객체 생성
llm = llm_connect()

if not os.getenv("LLM_API_KEY"):
    raise ValueError(
        "LLM_API_KEY가 설정되어 있지 않습니다. .env 파일 또는 환경변수에 OPENAI_API_KEY를 설정하세요."
    )

print("LLM_API_KEY 설정 확인 완료")

LLM_API_KEY 설정 확인 완료


## 3. PDF 파일 경로 설정

노트북과 같은 폴더 또는 `data` 폴더에 PDF를 넣어두면 자동으로 찾는다.

권장 폴더 구조는 다음과 같다.

```text
project/
├─ 1_faiss_naive_rag_financial_fraud.ipynb
└─ data/
   └─ 진화하는 금융사기, 모두의 경계가 필요한 시점.pdf
```

In [ ]:
PDF_NAME = "진화하는 금융사기, 모두의 경계가 필요한 시점.pdf"

candidate_paths = [
    Path(PDF_NAME),
    Path("data") / PDF_NAME,
    Path("/mnt/data") / PDF_NAME,  # ChatGPT 작업환경용 경로
]

PDF_PATH = next((path for path in candidate_paths if path.exists()), None)

if PDF_PATH is None:
    raise FileNotFoundError(
        f"PDF 파일을 찾을 수 없습니다. 다음 위치 중 하나에 파일을 넣어주세요: {candidate_paths}"
    )

print("PDF_PATH:", PDF_PATH)

PDF_PATH: data/진화하는 금융사기, 모두의 경계가 필요한 시점.pdf


## 4. PDF 로딩

Docling으로 PDF를 변환한 뒤 `DoclingLoader`가 `ExportType.DOC_CHUNKS`로 구조 기반 청크를 만든다.
PDF 파싱 조건(OCR 비활성화, 표 구조 인식, heading hierarchy)은 `02_document_processing_docling_chunks` 실습과 동일하게 맞춘다.

```text
PDF → DocumentConverter → DoclingDocument → HybridChunker → LangChain Document[]
```


### pdf 파일 로딩

In [ ]:
# ---------------------------------------------------------
# 1. Docling PDF 파싱 옵션
# ---------------------------------------------------------
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = False              # 텍스트 기반 PDF이므로 OCR 미사용
pipeline_options.do_table_structure = True   # 표의 행·열·셀 구조 인식
pipeline_options.heading_hierarchy_options = HeadingHierarchyOptions(enabled=True)
pipeline_options.generate_parsed_pages = True

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
    }
)

# ---------------------------------------------------------
# 2. HybridChunker 구성 (문서 구조 + token 한도)
# ---------------------------------------------------------
MAX_TOKENS = 700

# OpenAI embedding 계열 tokenizer로 청크의 token 수를 계산한다.
# 청킹 기준 계산용이며, 실제 임베딩은 common_config의 embedding_model()을 사용한다.
encoding = tiktoken.encoding_for_model("text-embedding-3-small")
tokenizer = OpenAITokenizer(tokenizer=encoding, max_tokens=MAX_TOKENS)

chunker = HybridChunker(
    tokenizer=tokenizer,
    merge_peers=True,            # 같은 heading 아래의 작은 인접 청크 병합
    repeat_table_header=True,    # 표가 나뉘어도 header 유지
)

print("DocumentConverter / HybridChunker 구성 완료")
print(f"Chunk 최대 token: {MAX_TOKENS}")


In [ ]:
# PDF → Docling 구조 기반 청크 (LangChain Document 리스트)
loader = DoclingLoader(
    file_path=str(PDF_PATH),
    converter=converter,
    export_type=ExportType.DOC_CHUNKS,
    chunker=chunker,
)

raw_chunks = loader.load()

print("생성된 Docling 청크 수:", len(raw_chunks))
print("첫 번째 청크 dl_meta 키:", list(raw_chunks[0].metadata.get("dl_meta", {}).keys()))


## 5. 문서 청킹 + Metadata 정규화

고정 길이 문자열로 자르지 않고, Docling이 인식한 제목·문단·표·caption 구조를 유지하면서 `HybridChunker`가 청크를 만든다.
Chroma에 저장하기 쉽도록 Docling 원본 metadata(`dl_meta`)에서 page, heading, caption만 평탄하게 꺼낸다.

- `page_label`: 청크의 대표 페이지 (Docling `page_no`, 1부터 시작)
- `pages`: 청크가 걸쳐 있는 전체 페이지
- `heading`: heading breadcrumb
- `caption`: 표/그림 caption
- `num_tokens`: 청크의 token 수


In [ ]:
def extract_pages(dl_meta: dict) -> list[int]:
    """Docling dl_meta의 provenance에서 페이지 번호를 추출한다."""
    pages = set()
    for item in dl_meta.get("doc_items", []):
        for prov in item.get("prov", []):
            page_no = prov.get("page_no")
            if page_no is not None:
                pages.add(page_no)
    return sorted(pages)


def normalize_docling_document(doc: Document, chunk_id: int) -> Document:
    """Docling metadata를 Chroma에서 다루기 쉬운 평탄한 metadata로 바꾼다."""
    dl_meta = doc.metadata.get("dl_meta", {})
    pages = extract_pages(dl_meta)
    headings = dl_meta.get("headings") or []
    captions = dl_meta.get("captions") or []

    # DOC_CHUNKS의 page_content는 이미 HybridChunker.contextualize() 결과이다.
    text = doc.page_content

    metadata = {
        "source_name": PDF_NAME,
        "chunk_id": chunk_id,

        # 기존 노트북과 같은 키로 페이지를 보관한다.
        "page_label": pages[0] if pages else -1,
        "pages": ",".join(map(str, pages)),
        "heading": " > ".join(map(str, headings)),
        "caption": " | ".join(map(str, captions)),
        "num_tokens": tokenizer.count_tokens(text),
        "parser": "docling",
        "chunker": "HybridChunker",
    }

    return Document(page_content=text, metadata=metadata)


chunks = [
    normalize_docling_document(doc, chunk_id=i)
    for i, doc in enumerate(raw_chunks)
]

print("정규화된 청크 수:", len(chunks))
print("첫 번째 청크 메타데이터:")
pprint(chunks[0].metadata)
print("\n첫 번째 청크 내용:")
print(chunks[0].page_content[:800])


## 6. 청크 분포 확인

페이지별 청크 수를 확인한다.  
특정 페이지의 청크가 지나치게 많거나 적으면 청킹 전략을 조정해야 한다.

In [ ]:
# 청크 정보를 DataFrame으로 정리하여 페이지별 청크 수와 샘플을 확인
chunk_df = pd.DataFrame([
    {
        "chunk_id": doc.metadata["chunk_id"],
        "page": doc.metadata.get("page_label"),
        "text_length": len(doc.page_content),
        "preview": doc.page_content[:80].replace("\n", " ")
    }
    for doc in chunks
])

print("페이지별 청크 수")
display(chunk_df.groupby("page").size().reset_index(name="chunk_count"))

print("청크 샘플")
display(chunk_df.head(10))

페이지별 청크 수


,page,chunk_count
0,1,1
1,2,2
2,3,2
3,4,2
4,5,2
5,6,2
6,7,2
7,8,2
8,9,2
9,10,2


청크 샘플


,chunk_id,page,text_length,preview
0,0,1,45,"2026년 5월 29일 제 17 호 진화하는 금융사기, 모두의 경계가 필요한 시점"
1,1,2,797,하나은행 ...
2,2,2,358,l 최근에는 AI로 목소리나 화면을 조작하여 피해자가 비대면으로 진위 여부를 확인할...
3,3,3,776,Hana Bank Hana Institute of Finance 2 ∎ 규제 공백 ...
4,4,3,266,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...
5,5,4,788,하나은행 ...
6,6,4,415,피해자가 범죄에 연루되는 것을 인지하지 못하도록 인력을 동원할 수 있다는 난점 ● ...
7,7,5,791,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
8,8,5,424,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | ..."
9,9,6,766,하나은행 ...


## 7. 임베딩 모델과 Chroma 벡터스토어 생성

여기서부터 인덱싱 단계이다.

```text
청크 텍스트 → 임베딩 벡터 → Chroma 컬렉션 저장
```

Chroma는 로컬 폴더에 영속 저장되는 벡터 DB이다.
이 노트북에서는 `Chroma`에 청크와 임베딩 모델을 연결하고, cosine distance를 사용하도록 설정한다.


In [ ]:
# 1. 임베딩 모델 초기화
embeddings = embedding_model()

# 2. Chroma 저장 경로와 컬렉션 설정
DB_PATH = Path("chroma_db/financial_fraud_pdf_chunks")
COLLECTION_NAME = "financial_fraud_pdf_docling_chunks"

# 3. 기존 컬렉션이 있으면 로딩, 없으면 새로 생성
#    (청킹 설정을 바꾸었다면 DB_PATH 폴더를 삭제한 뒤 다시 실행한다.)
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),

    # 검색 score를 cosine distance로 해석하도록 설정
    collection_metadata={"hnsw:space": "cosine"},
)

if vectorstore._collection.count() > 0:
    print("기존 Chroma 컬렉션을 로딩합니다.")
else:
    print("기존 Chroma 컬렉션이 없습니다.")
    print("Docling 청크를 임베딩하여 새 Chroma 컬렉션을 생성합니다.")
    vectorstore.add_documents(chunks)

print("Chroma 컬렉션 위치:", DB_PATH.resolve())
print("저장된 문서 청크 수:", vectorstore._collection.count())


## 8. 기본 유사도 검색 테스트

먼저 LLM 답변 생성 없이, 검색 결과만 확인한다.  
이 단계가 중요하다. 검색이 잘못되면 답변도 잘못될 가능성이 높다.

`similarity_search_with_score()`의 score는 Chroma에서 cosine distance로 계산된다.  
일반적으로 **낮을수록 더 가까운 문서**로 볼 수 있다.

In [ ]:
question = "금융사기는 어떤 단계로 발생하나요?"

results = vectorstore.similarity_search_with_score(question, k=4)

for rank, (doc, score) in enumerate(results, start=1):
    print(f"\n[{rank}] score={score:.4f} | page={doc.metadata.get('page_label')} | chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:500].replace("\n", " "))


[1] score=0.8513 | page=13 | chunk=24
● 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전반에  필요한 데이터를 종합적으로 분석하고 학습하여 사기를 예방할 수 있는 시스템 구축 ● 금융회사가 자체적으로 개발한 금융사기 탐지 프로그램을 타 금융사와 공유하는 등  사회적 문제 해결을 위한 공유 체계를 적극적으로 마련하여 피해를 예방하고자 노력 ■ 금융사기는 개인이 순간적으로 사기를 인지하고 예방하기 어렵다는 측면에서  금융회사와 관련 정부 기관의 예방 노력이 함께 이행되어야 함 ● 개별법 제·개정, 규제 강화 중심의 예방 노력은 후행적 조치라는 한계가 존재하므로   금융사기 예방을 위한 범기관 네트워크 구축이 요구 - 통신사, 가상자산거래소, 은행 등 금융사기가 발생하는 지점과 직접적으로 관련된 기관과의  공유 네트워크를 구축함으로써 민관 협력을 통한 예방 체계를 고도화 ● 금융사는 금융사기 전담 부서 조직을 확충하고 금융사기를 예방할 수 있는 시스템을  운영하는 한편, 피해 발생 시 배상 프

[2] score=0.8891 | page=4 | chunk=5
하나은행                                                                                                                    하나금융연구소 33 I. 금융사기란?  1. 금융사기의 정의와 특징 ■ 금융사기는 사람을 속이거나 착각하게 하여 금전적 이득을 취하는 범죄 행위 ● 금융사기는 사기 수법과 수단에 따라 양상이 다르게 나타나나, 공통적으로 ‘신뢰  유도-개인정보 수집-금전 편취’ 단계에 걸쳐 금전적 피해를 유발 - 택배회사 발송 배송조회 QR코드(신뢰)로 위장하여 피해자가 코드 스캔 시 악성코드나  앱을 스마트폰에 설치(수집)하고, 이를 통해 탈취한 개인정보로 금전을 유출(탈취) - 유명 핀플루언서(금융+인플루언서)의 영상을 도용(신뢰)하여 가짜 채널로 가

## 9. Retriever 구성

Retriever는 사용자의 질문을 받아 관련 문서 조각을 반환하는 객체이다.

```text
질문 → Retriever → 관련 청크 Top-k 반환
```

In [ ]:
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 4}
)

retrieved_docs = retriever.invoke("전기통신금융사기와 투자사기의 차이는 무엇인가요?")

for doc in retrieved_docs:
    print(f"page={doc.metadata.get('page_label')}, chunk={doc.metadata.get('chunk_id')}")
    print(doc.page_content[:300].replace("\n", " "))
    print("-" * 80)

page=7, chunk=12
- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹 안심차단(’25.11)을  차례로 시행하여 금융소비자가 자발적 신청을 통해 금융사기를 예방하도록 장려 ● 금융소비자가 직접 신청해야 하는 서비스 외에 자금 이체 시 수취인 계좌에 일정 시간  경과 후 입금되는 지연이체 서비스 등 부정출금 방지를 위한 거래 방식을 도입 표3 | 전기통신금융사기 주요 대응 현황 표4 | 투자사기 주요 대응 현황 시기 주요 내용 시기 주요 내용 ’23.7 통합신고대응센터 개소 ’24.1 자본시장법  개정  Ÿ 불공정거래 
--------------------------------------------------------------------------------
page=5, chunk=7
Hana Bank Hana Institute of Finance 4 2. 금융사기 주요 유형 ■ 금융사기는 금전적 피해를 유발하는 범죄행위 전반을 포괄해 다양한 기준으로  구분되나 전기통신금융사기와 투자사기 두 분류로 구분하는 것이 보편적 ● 전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신을 이용하여 금융기관·    지인·가족 등을 사칭하거나 허위 사실로 피해자의 자금을 탈취하는 수법 - 피싱, 스미싱, 파밍 등이 대표적인 유형이며, 법에서 열거하는 통신수단을 이용하여   개인정보를 노출시켜 자금을 송금·이체하도록 유도 - ’
--------------------------------------------------------------------------------
page=5, chunk=8
사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | 금융사기 분류 그림2 | 금융사기 접근 경로 분류 하위 종류 전기통신 금융사기 보이스피싱 전화로 타인을 사칭하여 송금을 유도하는 수법 메신저피싱 카카오톡이나 SNS로 타인을 사칭하여 송금을  유도하는 수법 파밍 가짜 사이트에 접속하여 개인정보를 입력하도

## 10. RAG 답변 프롬프트 구성

핵심은 LLM이 검색된 문서 밖의 내용을 임의로 생성하지 않도록 제약하는 것이다.

프롬프트 설계 원칙은 다음과 같다.

1. 문서 근거만 사용한다.
2. 문서에서 확인할 수 없으면 모른다고 답한다.
3. 답변 마지막에 근거 페이지를 포함한다.

In [ ]:
def format_docs(docs):
    """검색된 문서들을 LLM 입력용 context 문자열로 변환한다."""
    formatted = []
    for doc in docs:
        page = doc.metadata.get("page_label", "?")
        chunk_id = doc.metadata.get("chunk_id", "?")
        formatted.append(
            f"[출처: page {page}, chunk {chunk_id}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """
당신은 금융사기 보고서 기반 QA 어시스턴트입니다.
아래 [문서]에 근거해서만 답변하세요.
문서에서 확인할 수 없는 내용은 '문서에서 확인할 수 없습니다.'라고 답변하세요.
답변은 핵심 위주로 3~5문장으로 정리하세요.
답변 마지막에 근거 페이지를 '근거: p.숫자' 형식으로 표시하세요.

[문서]
{context}
""".strip()
    ),
    ("human", "{question}"),
])

llm = llm_connect()

answer_chain = prompt | llm | StrOutputParser()

## 11. RAG 함수 만들기

검색과 답변 생성을 하나의 함수로 묶는다.

```text
질문 입력
→ Chroma 검색
→ 검색 문서 format
→ LLM 답변 생성
→ 답변과 출처 반환
```

In [ ]:
def answer_question(question: str, k: int = 4):
    """질문에 대해 Chroma 검색 후 문서 기반 답변과 출처 테이블을 반환한다."""
    docs = vectorstore.similarity_search(question, k=k)
    context = format_docs(docs)

    answer = answer_chain.invoke({
        "question": question,
        "context": context,
    })

    sources = pd.DataFrame([
        {
            "rank": i + 1,
            "page": doc.metadata.get("page_label"),
            "chunk_id": doc.metadata.get("chunk_id"),
            "preview": doc.page_content[:120].replace("\n", " ")
        }
        for i, doc in enumerate(docs)
    ])

    return answer, sources

## 12. RAG 실행 예시

다음 질문들은 이 보고서에 포함된 내용을 기반으로 답변 가능한 질문이다.

In [ ]:
question = "전기통신금융사기와 투자사기는 어떻게 다른가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)

print("\n[검색된 근거]")
display(sources)  # Jupyter 환경에서는 display로 테이블이 예쁘게 나옵니다.


[질문]
전기통신금융사기와 투자사기는 어떻게 다른가요?

[답변]
전기통신금융사기는 전화, 메신저, 인터넷 등 전기통신수단을 이용해 금융기관·지인·가족 등을 사칭하거나 허위 사실로 자금을 탈취하는 범죄입니다. 보이스피싱, 메신저피싱, 파밍, 스미싱, 해킹 등이 대표적이며, 송금·이체를 유도하는 방식이 특징입니다.  
반면 투자사기는 투자자의 수익 기대를 이용해 투자를 유도하거나 가격을 조작해 금전적 피해를 주는 범죄로, 리딩방 사기나 비상장주식 과장 판매, 가짜 HTS/MTS 유도가 포함됩니다. 즉, 전기통신금융사기는 “사칭·기망을 통한 자금 탈취”가 중심이고, 투자사기는 “투자 명목의 유인과 수익 미끼”가 중심입니다.  
근거: p.5

[검색된 근거]


,rank,page,chunk_id,preview
0,1,5,7,Hana Bank Hana Institute of Finance 4 2. 금융사기 ...
1,2,5,8,"사기 행위를 분류하며, 수법에 따라 관련 규정을 지속적으로 확대하는 추세 표2 | ..."
2,3,7,12,"- 여신거래 안심차단(’24.8), 비대면 계좌개설 안심차단(’25.3), 오픈뱅킹..."
3,4,3,4,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...


In [ ]:
question = "금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
금융회사들은 금융사기를 예방하기 위해 어떤 시스템을 도입하고 있나요?

[답변]
금융회사들은 비정상적 금융거래를 실시간으로 분석해 부정거래를 탐지·차단하는 이상금융거래탐지시스템(FDS)을 도입하고 있습니다. 또한 거래 내역, 고객 정보, 평소 거래 패턴을 분석해 이상 거래를 막고, 전담 조직을 확충하거나 사기 방지 소프트웨어를 자체 개발하는 등 구조적 대응도 강화하고 있습니다. 더 나아가 생성형 AI를 접목해 신종 사기 수법을 학습·탐지하거나, 통신사 등 관련 기관과 협업해 실시간 예방 체계를 마련하고 있습니다. 근거: p.9, p.13

[검색된 근거]


,rank,page,chunk_id,preview
0,1,13,24,"● 금융거래 데이터뿐만 아니라 앱 이용 패턴 데이터, 통신 데이터 등 사기 거래 전..."
1,2,3,4,측면에서 금융회사와 정부 기관의 예방 노력이 함께 이행되어야 할 필요 l 개별법 제...
2,3,9,15,Hana Bank Hana Institute of Finance 8 2. 금융회사 ...
3,4,12,22,"- 거래 이력, 계좌 이름, 결제 금액 등 사기 계좌를 탐지할 수 있는 정보를 종합..."


## 13. 문서에 없는 질문 테스트

RAG 실습에서는 답변 가능한 질문만 넣으면 안 된다.  
문서에 없는 질문을 넣어 **모른다고 답하는지** 확인해야 한다.

In [ ]:
question = "2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?"

answer, sources = answer_question(question, k=4)

print("[질문]")
print(question)
print("\n[답변]")
print(answer)
print("\n[검색된 근거]")
display(sources)

[질문]
2026년 6월 기준 최신 보이스피싱 피해 금액은 얼마인가요?

[답변]
문서에서 확인되는 최신 수치는 ’24년 기준 보이스피싱·유사수신 등 불특정 다수를 대상으로 한 금융사기 피해금액 약 1.7조 원입니다. 다만 질문하신 **2026년 6월 기준** 최신 보이스피싱 피해 금액은 문서에 제시되어 있지 않아 확인할 수 없습니다. 문서에는 ’25년 관련 수치나 ’26년 6월 기준 피해금액도 없습니다. 근거: p.6

[검색된 근거]


,rank,page,chunk_id,preview
0,1,6,10,● 연애 관계를 빙자하여 자금을 탈취하는 로맨스스캠의 경우 ’24년 2~12월 기준...
1,2,10,18,- 금융사기의 수법이 진화하여 소비자 개인이 사기를 예방하는 데 실질적 한계가 있다...
2,3,6,9,하나은행 ...
3,4,10,17,하나은행 ...


## 14. 정리

이 실습에서 구현한 Naive RAG 흐름은 다음과 같다.

```text
PDF 로딩 (DoclingLoader, ExportType.DOC_CHUNKS)
→ Docling 구조 기반 청크 생성 (HybridChunker, 700 token)
→ metadata 정규화 (page_label, heading, caption)
→ 임베딩 (common_config의 embedding_model)
→ Chroma 벡터스토어 생성
→ 질문 기반 Top-k 검색
→ 검색 문서 기반 LLM 답변 생성
```

## 핵심 포인

| 관찰 포인트 | 핵심 내용 |
|---|---|
| 검색 결과가 부정확함 | 청크 크기, 질문 표현, Top-k 조정 필요 |
| 답변이 문서 밖 내용을 포함함 | 프롬프트 제약 및 문서 없음 질문 테스트 필요 |
| 표·그림 내용이 누락됨 | PDF 파싱 방식의 한계, Markdown 변환 비교 필요 |
| 같은 질문인데 답변이 달라짐 | 검색 결과와 LLM 생성 조건을 분리해서 점검해야 함 |
| 인덱스 재사용 필요 | Chroma 영속 저장 구조로 인덱싱 비용 절감 가능 |